# Crear una voz propia para Locutor

Este cuaderno crea una voz Piper nueva a partir de un audio corto de referencia y la deja lista para **Importar voz** en Locutor.

**Cómo funciona**
1. **Parte 1 – Datos:** Chatterbox (MIT) clona la voz de tu audio y lee cientos de frases en español.
2. **Parte 2 – Entrenamiento:** se ajusta un modelo Piper con esas frases (Piper1-GPL, método oficial).
3. **Parte 3 – Exportar:** se generan el `.onnx` y el `.onnx.json` para importar en Locutor.

**Antes de empezar**
- Menú *Entorno de ejecución → Cambiar tipo de entorno* → **GPU T4**.
- Usá solo **tu propia voz** o la de alguien que te haya dado permiso.
- Todo se guarda en tu Google Drive, en `locutor_voces/`, así que si Colab se corta podés retomar.
- Entre la Parte 1 y la Parte 2 hay que **reiniciar el entorno** (las librerías chocan entre sí).

Tiempo aproximado con la GPU gratuita: 1 a 2 h para los datos y varias horas de entrenamiento, repartidas en varias sesiones si hace falta.

## Configuración (correr siempre primero, también después de reiniciar)

In [ ]:
#@title Datos de la voz
NOMBRE = "mi_voz"        #@param {type:"string"}
REGION = "es_AR"         #@param ["es_AR", "es_MX", "es_ES"]
FRASES = 500             #@param {type:"integer"}
EPOCAS = 800             #@param {type:"integer"}

import os, re
from google.colab import drive
drive.mount('/content/drive')
BASE = f"/content/drive/MyDrive/locutor_voces/{NOMBRE}"
WAVS = f"{BASE}/wavs"
os.makedirs(WAVS, exist_ok=True)
ESPEAK = "es" if REGION == "es_ES" else "es-419"
print("Carpeta de trabajo:", BASE)

# Parte 1 – Generar los datos

Subí un audio de **10 a 20 segundos**, con una sola persona hablando claro, sin música ni ruido. WAV o MP3.

In [ ]:
#@title 1.1 Instalar Chatterbox
!pip -q install chatterbox-tts

In [ ]:
#@title 1.2 Subir el audio de referencia
from google.colab import files
import shutil
subido = files.upload()
nombre_archivo = next(iter(subido))
REF = f"{BASE}/referencia{os.path.splitext(nombre_archivo)[1]}"
shutil.move(nombre_archivo, REF)
print("Referencia guardada en", REF)

In [ ]:
#@title 1.3 Juntar frases en español
# Frases de Tatoeba (licencia CC BY 2.0 FR). Se eligen frases de largo medio y variado.
import bz2, random, urllib.request, glob
random.seed(7)
frases_txt = f"{BASE}/frases.txt"
if not os.path.exists(frases_txt):
    url = "https://downloads.tatoeba.org/exports/per_language/spa/spa_sentences.tsv.bz2"
    urllib.request.urlretrieve(url, "/content/spa.tsv.bz2")
    todas = []
    with bz2.open("/content/spa.tsv.bz2", "rt", encoding="utf-8") as f:
        for linea in f:
            partes = linea.rstrip("\n").split("\t")
            if len(partes) >= 3:
                t = partes[2].strip()
                if 35 <= len(t) <= 170 and not re.search(r"[0-9\"«»()\[\]/]", t):
                    todas.append(t)
    random.shuffle(todas)
    with open(frases_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(todas[: max(FRASES * 2, 1000)]))
frases = open(frases_txt, encoding="utf-8").read().splitlines()
print(len(frases), "frases disponibles. Ejemplo:", frases[0])

In [ ]:
#@title 1.4 Generar los audios (se puede cortar y retomar)
import torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

REF = sorted(glob.glob(f"{BASE}/referencia.*"))[0]
modelo = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
meta_path = f"{BASE}/metadata.csv"
hechas = {}
if os.path.exists(meta_path):
    for l in open(meta_path, encoding="utf-8"):
        archivo, texto = l.rstrip("\n").split("|", 1)
        hechas[archivo] = texto

i = len(hechas)
with open(meta_path, "a", encoding="utf-8") as meta:
    for texto in frases:
        if i >= FRASES:
            break
        archivo = f"f{i:05d}.wav"
        if archivo in hechas:
            i += 1
            continue
        try:
            wav = modelo.generate(texto, language_id="es", audio_prompt_path=REF)
        except Exception as e:
            print("Salteo una frase:", e)
            continue
        wav = torchaudio.functional.resample(wav.cpu(), modelo.sr, 22050)
        dur = wav.shape[-1] / 22050
        if dur < 1.0 or dur > 15:
            continue
        torchaudio.save(f"{WAVS}/{archivo}", wav, 22050)
        meta.write(f"{archivo}|{texto}\n"); meta.flush()
        i += 1
        if i % 25 == 0:
            print(f"{i}/{FRASES} frases listas")
print("Listo:", i, "frases en", WAVS)

In [ ]:
#@title 1.5 Escuchar un par de ejemplos
from IPython.display import Audio, display
for l in open(f"{BASE}/metadata.csv", encoding="utf-8").read().splitlines()[:3]:
    archivo, texto = l.split("|", 1)
    print(texto); display(Audio(f"{WAVS}/{archivo}"))

### ⚠️ Ahora reiniciá el entorno
Menú *Entorno de ejecución → Reiniciar sesión*. Después corré otra vez la celda de **Configuración** y seguí con la Parte 2.

# Parte 2 – Entrenar la voz

In [ ]:
#@title 2.1 Instalar Piper (entrenamiento)
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
%cd /content
!rm -rf piper1-gpl && git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!bash build_monotonic_align.sh
!python3 setup.py build_ext --inplace

In [ ]:
#@title 2.2 Bajar un punto de partida (checkpoint "medium")
# Se prefiere uno en español; si no hay, se usa Lessac (inglés), que igual acelera mucho.
from huggingface_hub import list_repo_files, hf_hub_download
archivos = list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset")
ckpts = [f for f in archivos if f.endswith(".ckpt") and "/medium/" in f]
preferidos = [f for f in ckpts if f.startswith(f"es/{REGION}/")] or [f for f in ckpts if f.startswith("es/")] \
             or [f for f in ckpts if "en_US/lessac/medium" in f]
print("Opciones:", preferidos[:5])
BASE_CKPT = hf_hub_download("rhasspy/piper-checkpoints", preferidos[0], repo_type="dataset")
print("Usando:", preferidos[0])

In [ ]:
#@title 2.3 Entrenar (si se corta, volvé a correr 2.1, 2.2 y esta celda: retoma solo)
import glob, subprocess
RAIZ = f"{BASE}/entrenamiento"
ultimos = sorted(glob.glob(f"{RAIZ}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)
inicio = ["--ckpt_path", ultimos[-1]] if ultimos else ["--model.warmstart_ckpt", BASE_CKPT]
print("Retomando desde" if ultimos else "Empezando desde", inicio[1])
cmd = ["python3", "-m", "piper.train", "fit",
       "--data.voice_name", NOMBRE,
       "--data.csv_path", f"{BASE}/metadata.csv",
       "--data.audio_dir", WAVS,
       "--model.sample_rate", "22050",
       "--data.espeak_voice", ESPEAK,
       "--data.cache_dir", "/content/cache",
       "--data.config_path", f"{BASE}/config.json",
       "--data.batch_size", "16",
       "--trainer.max_epochs", str(EPOCAS),
       "--trainer.default_root_dir", RAIZ] + inicio
subprocess.run(cmd, cwd="/content/piper1-gpl", check=True)

# Parte 3 – Exportar para Locutor

In [ ]:
#@title 3.1 Exportar y descargar
import glob, shutil
from google.colab import files
ckpt = sorted(glob.glob(f"{BASE}/entrenamiento/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)[-1]
salida = f"{BASE}/{REGION}-{NOMBRE}-medium.onnx"
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file "{salida}"
shutil.copy(f"{BASE}/config.json", salida + ".json")
print("Archivos listos en Drive:", salida, "y", salida + ".json")
files.download(salida); files.download(salida + ".json")

## Usar la voz en Locutor
1. Abrí Locutor y bajá hasta **Importar voz**.
2. Tocá **Elegir archivos del dispositivo** y seleccioná los dos archivos juntos (`.onnx` y `.onnx.json`).
3. La voz queda guardada y seleccionada. Generá un texto para escucharla.

Si suena con poca calidad, generá más frases (800 a 1000) y entrená más épocas.